# Mercado: Lieferqualität und Kundenservice
 **DABI1 · Projektaufgabe**

Gruppe | Studierende 1 | Studierende 2 | Studierende 3 |
------ | ------ | ------ |-----|
`8` | `Damian Frehar` | `Nils Hahn`|`Maximilian Hutter` |

## 2. Ihr Beratungsauftrag

### 2.1 Daten vorbereiten

#### Import CSV Dateien und Python Bibliotheken

In [ ]:
import pprint

import pandas as pd
import plotly.express as px
from pathlib import Path
import pprint as pp

In [ ]:
# Dictionary für die DataFrames
dataframes = {}

data_folder = Path("data")

# Alle CSV-Dateien im Ordner durchlaufen
for file in data_folder.glob("*.csv"):
    df = pd.read_csv(file)
    dataframes[file.stem] = df

#### Datentypen Prüfen

In [ ]:
for i in dataframes:
    print(i)
    types = dataframes[i].dtypes
    print(types)
    print("")

#### Fehlende Werte prüfen

In [ ]:
for i in dataframes:
    print(i)
    print(dataframes[i].isna().sum())
    print("")

In [ ]:
import missingno as msno

for i in dataframes:
    print(i)
    msno.matrix(dataframes[i])

#### Schlüssel prüfen

In [ ]:
dataframes["products"].nunique()

for i in dataframes:
    print(i)
    print(dataframes[i].nunique())
    print(dataframes[i].info())
    print("")

In [ ]:
dataframes["products"].info()
for i in dataframes:
    print(i)
    msno.matrix(dataframes[i])

In [ ]:
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'



for i in dataframes:
    schluessel = [
        col for col in dataframes[i].columns
        if dataframes[i][col].notna().all()
        and dataframes[i][col].is_unique
    ]


    if schluessel == []:
        print(bcolors.WARNING + f"Schlüssel für {i} ist die Spalte: {schluessel}")
    else:
        print(bcolors.OKBLUE + f"Schlüssel für {i} ist die Spalte: {schluessel}")

In [ ]:
dataframes["payments"]

In [ ]:
dataframes["payments"][
    dataframes["payments"].duplicated(keep=False)
]

In [ ]:
no_duplicates = dataframes["payments"].drop_duplicates(keep=False)
result = dataframes["payments"].value_counts().reset_index(name='count')
result = result[result['count'] > 1]

result.info()

In [ ]:
for name, df in dataframes.items():
    print(f'{name}: {df.index.duplicated().sum()} Duplikate')

In [ ]:
a = 128
print("a")
print("SOmetimes I dream of saving the world")

In [ ]:
px.scatter(dataframes["payments"], x = "payment_type", height=1000, y = "payment_value", color = "payment_installments", hover_data="order_id")

In [ ]:
df = dataframes["products"]

df.loc[
    df["product_category_name"].isna() & df["product_photos_qty"].notna() +
    df["product_photos_qty"].isna() & df["product_category_name"].notna()
]

In [ ]:
cat = dataframes["categories"]
cat
prod_merge = pd.merge(df, cat, left_on="product_category_name", right_on="product_category_name")
prod_merge

In [ ]:
prod_merge["volume"] = prod_merge["product_height_cm"] * prod_merge["product_length_cm"] * df["product_width_cm"]
prod_merge["weight_kg"] = prod_merge["product_weight_g"]/1000

#df_clean = prod_merge[prod_merge["weight_kg"] < 100]

px.scatter(prod_merge, x="volume", y="weight_kg", 
            hover_data="product_category_name_english", 
            color="product_category_name_english", 
            title= "Gewicht und Volumen Verteilung der Produkte", log_y=True)

In [ ]:

df[df["product_category_name"] == "automotivo"].sort_values(by="product_weight_g", ascending=False).head(10)

# Untersuchung von orders.csv

In [ ]:
orders = dataframes["orders"]
orders

In [ ]:
orders.dtypes

In [ ]:
orders.head()

# Datentypen
(Konvertierung in Datatime)

In [ ]:
orders = dataframes["orders"]

datumsspalten = ["order_purchase_timestamp", "order_approved_at",
                 "order_delivered_carrier_date", "order_delivered_customer_date",
                 "order_estimated_delivery_date"]

for spalte in datumsspalten:
    orders[spalte] = pd.to_datetime(orders[spalte], format="%Y-%m-%d %H:%M:%S")

orders.dtypes

# Schlüssel

In [ ]:
schluessel = [
        col for col in orders.columns
        if orders[col].notna().all()
        and orders[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für orders ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für orders ist die Spalte: {schluessel}")

# Fehlende Werte

In [ ]:
msno.matrix(orders)

In [ ]:
orders[orders["order_status"] == "canceled"]

orders["isMissing"] = orders["order_approved_at"].isna()


Wenn order_status "cancelled" ist, dann fehlt immer order_delivered_customer_date und auch meistens order_delivered_carrier_date (macht ja auch Sinn)

In [ ]:
px.histogram(orders, y="order_status", color="isMissing", width=1000)

3 Zeilen haben ein fehlendes order_approved_at, dies liegt daran da eine order erst auf created liegt und zwei wurden direkt gecanceled

# plausible Größenordnungen

In [ ]:
for spalte in datumsspalten:
    print(spalte)
    print(orders[spalte].min())
    print(orders[spalte].max())
    print("")

Kategorien von order status

In [ ]:
orders["order_status"].unique()

Fazit:
Datumspalten wurden in Datetime umgewandelt,
Fehlende Werte bei orders werden beibehalten, da die Quantität der fehlenden Daten überschauber ist, und die Werte aus einem plausiblen Grund fehlen,


# customers.csv

In [ ]:
customers = dataframes["customers"]

# Datentypen

In [ ]:
customers.dtypes

In [ ]:
customers.head()

In [ ]:
customers["customer_city"].unique()

In [ ]:
customers["customer_state"].unique()

# Schlüssel

In [ ]:
schluessel = [
        col for col in customers.columns
        if customers[col].notna().all()
        and customers[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für customers ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für customers ist die Spalte: {schluessel}")

macht sinn weil customer_unique_id ist mehrmals vorhanden da Kunden mehrere Bestellungen machen können

# Fehlende Werte

In [ ]:
msno.matrix(customers)

keine fehlenden Werte :)

In [ ]:
a = 123
print("a")
print("Sometimes I dream of saving the world")

# Plausible Ordnungsgrößen

In [ ]:
px.histogram(customers, x="customer_state", color="customer_city")

sao paulo macht sinn weil das die größte Stadt ist

# items.csv

# Datentypen

In [ ]:
items = dataframes["items"]

In [ ]:
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], format="%Y-%m-%d %H:%M:%S")

In [ ]:
items.dtypes

In [ ]:
items

# Schlüssel

In [ ]:
schluessel = [
        col for col in items.columns
        if items[col].notna().all()
        and items[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für items ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für items ist die Spalte: {schluessel}")

In [ ]:
items["order_id"].nunique()

In [ ]:
items["order_key"] = items["order_id"].astype(str) + "_" + items["order_item_id"].astype(str)

In [ ]:
items["order_key"].nunique()

In [ ]:
items["order_key"].duplicated(keep=False)

In [ ]:
items["order_key"].duplicated().sum()

In [ ]:
items[items["order_key"].duplicated(keep=False)].sort_values("order_key")

In [ ]:
items["order_key"].value_counts()[lambda x: x > 1]

In [ ]:
items.duplicated().sum()

Wir haben 225 Zeilen welche identische Duplikate sind, raus mit denen!

In [ ]:
items = items.drop_duplicates()

In [ ]:
items.duplicated().sum()

In [ ]:
schluessel = [
        col for col in items.columns
        if items[col].notna().all()
        and items[col].is_unique
    ]


if schluessel == []:
    print(bcolors.WARNING + f"Schlüssel für items ist die Spalte: {schluessel}")
else:
    print(bcolors.OKBLUE + f"Schlüssel für items ist die Spalte: {schluessel}")

# Fehlende Werte

In [ ]:
msno.matrix(items)

keine fehlenden Werte in items.csv

# Plausible Ordnungsgrößen

Klären wir erstmal das offensichtlichste Vorab. Warum trag ich nh Brille?

Keine Ahnung...

Ist auch völlig uninteressant

In [ ]:
px.histogram(items, x="price")

In [ ]:
items["price"].describe()

In [ ]:
#Untersuchen was das teuerste Produkt ist
items.sort_values(by="price", ascending=False)

In [ ]:
products = dataframes["products"]
products[products["product_id"] == "489ae2aa008f021502940f251d4cce7f"]

joa wahrscheinlich passt des schon

In [ ]:
px.histogram(items, x="freight_value")

In [ ]:
items['freight_value'].describe()

In [ ]:
items["shipping_limit_date"].max()
items["shipping_limit_date"].min()